In [9]:
import os
import json
import copy
import time
from tqdm import tqdm
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [10]:
# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

DATASET_PATH = r"D:\PlantDiseaseDataset"
TRAIN_DIR = os.path.join(DATASET_PATH, "train")
VALID_DIR = os.path.join(DATASET_PATH, "valid")

OUTPUT_DIR = os.path.join(DATASET_PATH, "saved_model")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_PATH = os.path.join(
    OUTPUT_DIR, "plant_disease_mobilenetv2.pth"
)
LABELS_PATH = os.path.join(
    OUTPUT_DIR, "class_labels.json"
)
PLOT_PATH = os.path.join(OUTPUT_DIR, "training_history.png")

IMAGE_SIZE = 224
BATCH_SIZE = 32
INITIAL_EPOCHS = 5
FINE_TUNE_EPOCHS = 5

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [11]:
# --------------------------------------------------
# 2. Image transformations
# --------------------------------------------------

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.15, contrast=0.15, saturation=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

valid_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [12]:

# --------------------------------------------------
# 3. Load datasets
# --------------------------------------------------

train_dataset = datasets.ImageFolder(
    TRAIN_DIR, transform=train_transform
)

valid_dataset = datasets.ImageFolder(
    VALID_DIR, transform=valid_transform
)

# ImageFolder assigns class indices alphabetically.
class_names = train_dataset.classes

if valid_dataset.classes != class_names:
    raise ValueError(
        "Training and validation class folders do not match."
    )

num_classes = len(class_names)

if num_classes < 2:
    raise ValueError("At least two classes are required.")

print("Number of classes:", num_classes)
print("Classes:", class_names)
print("Training images:", len(train_dataset))
print("Validation images:", len(valid_dataset))

with open(LABELS_PATH, "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=4, ensure_ascii=False)



Number of classes: 9
Classes: ['Apple', 'Cherry', 'Corn', 'Grape', 'Peach', 'Pepper', 'Potato', 'Strawberry', 'Tomato']
Training images: 49647
Validation images: 12494


In [13]:

# --------------------------------------------------
# 4. Data loaders
# --------------------------------------------------

# Reduce workers if Windows multiprocessing causes issues.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)


In [14]:

# --------------------------------------------------
# 5. Load pretrained MobileNetV2
# --------------------------------------------------

weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)

# Freeze the pretrained feature extractor initially.
for param in model.features.parameters():
    param.requires_grad = False

# Replace the final classification layer.
in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    in_features, num_classes
)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = AdamW(
    model.classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = ReduceLROnPlateau(
    optimizer, mode="min", factor=0.3, patience=2
)


In [15]:

# --------------------------------------------------
# 6. Training and validation function
# --------------------------------------------------

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total



In [16]:
best_val_loss = float("inf")
best_state = copy.deepcopy(model.state_dict())

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}


def fit(epochs, phase_name):
    global best_val_loss, best_state

    patience_counter = 0
    early_stop_patience = 4

    for epoch in range(epochs):
        epoch_start = time.time()

        print(
            f"\n{'=' * 60}\n"
            f"{phase_name} | Epoch {epoch + 1}/{epochs}\n"
            f"{'=' * 60}",
            flush=True
        )

        # ---------------------------
        # Training
        # ---------------------------
        model.train()

        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        train_bar = tqdm(
            train_loader,
            desc="Training",
            unit="batch",
            dynamic_ncols=True
        )

        for images, labels in train_bar:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)

            train_loss_sum += loss.item() * batch_size
            predictions = outputs.argmax(dim=1)
            train_correct += (predictions == labels).sum().item()
            train_total += batch_size

            current_loss = train_loss_sum / train_total
            current_acc = train_correct / train_total

            train_bar.set_postfix(
                loss=f"{current_loss:.4f}",
                accuracy=f"{current_acc:.2%}"
            )

        train_loss = train_loss_sum / train_total
        train_acc = train_correct / train_total

        # ---------------------------
        # Validation
        # ---------------------------
        model.eval()

        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            valid_bar = tqdm(
                valid_loader,
                desc="Validation",
                unit="batch",
                dynamic_ncols=True
            )

            for images, labels in valid_bar:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)

                outputs = model(images)
                loss = criterion(outputs, labels)

                batch_size = labels.size(0)

                val_loss_sum += loss.item() * batch_size
                predictions = outputs.argmax(dim=1)
                val_correct += (
                    predictions == labels
                ).sum().item()
                val_total += batch_size

                valid_bar.set_postfix(
                    loss=f"{val_loss_sum / val_total:.4f}",
                    accuracy=f"{val_correct / val_total:.2%}"
                )

        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total

        # Update learning rate
        scheduler.step(val_loss)

        # Save history
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        elapsed = time.time() - epoch_start

        print(f"\nEpoch {epoch + 1}/{epochs} completed")
        print(f"Train Loss      : {train_loss:.4f}")
        print(f"Train Accuracy  : {train_acc:.2%}")
        print(f"Validation Loss : {val_loss:.4f}")
        print(f"Validation Acc  : {val_acc:.2%}")
        print(f"Time taken      : {elapsed / 60:.2f} minutes")

        # ---------------------------
        # Save best model
        # ---------------------------
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

            torch.save({
                "model_state_dict": best_state,
                "class_names": class_names,
                "num_classes": num_classes,
                "image_size": IMAGE_SIZE
            }, MODEL_PATH)

            patience_counter = 0
            print("Best model saved!", flush=True)

        else:
            patience_counter += 1
            print(
                f"No validation loss improvement: "
                f"{patience_counter}/{early_stop_patience}"
            )

        if patience_counter >= early_stop_patience:
            print("Early stopping triggered.")
            break


# Phase 1: Train classifier
print("\nPhase 1: Training classifier", flush=True)
fit(INITIAL_EPOCHS, "Classifier")



Phase 1: Training classifier

Classifier | Epoch 1/5


Validation: 100%|██████████| 391/391 [02:40<00:00,  2.44batch/s, accuracy=94.05%, loss=0.2155]


Epoch 1/5 completed
Train Loss      : 0.3898
Train Accuracy  : 90.17%
Validation Loss : 0.2155
Validation Acc  : 94.05%
Time taken      : 12.54 minutes
Best model saved!

Classifier | Epoch 2/5



Validation: 100%|██████████| 391/391 [00:28<00:00, 13.66batch/s, accuracy=95.49%, loss=0.1562]



Epoch 2/5 completed
Train Loss      : 0.1862
Train Accuracy  : 94.27%
Validation Loss : 0.1562
Validation Acc  : 95.49%
Time taken      : 7.27 minutes
Best model saved!

Classifier | Epoch 3/5


...

Validation: 100%|██████████| 391/391 [00:38<00:00, 10.27batch/s, accuracy=95.29%, loss=0.1464]


Epoch 4/5 completed
Train Loss      : 0.1432
Train Accuracy  : 95.23%
Validation Loss : 0.1464
Validation Acc  : 95.29%
Time taken      : 7.80 minutes
Best model saved!



Classifier | Epoch 5/5


Validation: 100%|██████████| 391/391 [00:40<00:00,  9.77batch/s, accuracy=96.19%, loss=0.1255]



Epoch 5/5 completed
Train Loss      : 0.1398
Train Accuracy  : 95.34%
Validation Loss : 0.1255
Validation Acc  : 96.19%
Time taken      : 7.67 minutes
Best model saved!


In [17]:

# # --------------------------------------------------
# # 7. Training loop with best-model saving
# # --------------------------------------------------

# best_val_loss = float("inf")
# best_state = copy.deepcopy(model.state_dict())
# history = {
#     "train_loss": [],
#     "val_loss": [],
#     "train_acc": [],
#     "val_acc": []
# }

# def fit(epochs, phase_name):
#     global best_val_loss, best_state

#     patience_counter = 0
#     early_stop_patience = 4

#     for epoch in range(epochs):
#         train_loss, train_acc = run_epoch(
#             model, train_loader, optimizer
#         )

#         val_loss, val_acc = run_epoch(
#             model, valid_loader
#         )

#         scheduler.step(val_loss)

#         history["train_loss"].append(train_loss)
#         history["val_loss"].append(val_loss)
#         history["train_acc"].append(train_acc)
#         history["val_acc"].append(val_acc)

#         print(
#             f"{phase_name} | Epoch {epoch + 1}/{epochs} | "
#             f"Train Loss: {train_loss:.4f} | "
#             f"Train Acc: {train_acc:.2%} | "
#             f"Val Loss: {val_loss:.4f} | "
#             f"Val Acc: {val_acc:.2%}"
#         )

#         if val_loss < best_val_loss:
#             best_val_loss = val_loss
#             best_state = copy.deepcopy(model.state_dict())

#             torch.save({
#                 "model_state_dict": best_state,
#                 "class_names": class_names,
#                 "num_classes": num_classes,
#                 "image_size": IMAGE_SIZE
#             }, MODEL_PATH)

#             patience_counter = 0
#             print("Best model saved.")
#         else:
#             patience_counter += 1

#         if patience_counter >= early_stop_patience:
#             print("Early stopping.")
#             break

# # Phase 1: Train classification head.
# print("\nPhase 1: Training classifier")
# fit(INITIAL_EPOCHS, "Classifier")


In [18]:

# # --------------------------------------------------
# # 8. Fine-tune final MobileNetV2 feature layers
# # --------------------------------------------------

# print("\nPhase 2: Fine-tuning")

# for param in model.features.parameters():
#     param.requires_grad = False

# # Unfreeze the final feature blocks.
# for param in model.features[-3:].parameters():
#     param.requires_grad = True

# optimizer = AdamW(
#     filter(lambda p: p.requires_grad, model.parameters()),
#     lr=1e-5,
#     weight_decay=1e-4
# )

# scheduler = ReduceLROnPlateau(
#     optimizer, mode="min", factor=0.3, patience=2
# )

# fit(FINE_TUNE_EPOCHS, "Fine-tuning")

# # Restore the best weights found across both phases.
# model.load_state_dict(best_state)
# model.eval()


In [19]:

# --------------------------------------------------
# 8. Fine-tune MobileNetV2 with tqdm progress
# --------------------------------------------------

from tqdm import tqdm
import time

print("\nPhase 2: Fine-tuning", flush=True)

# Freeze all feature layers
for param in model.features.parameters():
    param.requires_grad = False

# Unfreeze the final three feature blocks
for param in model.features[-3:].parameters():
    param.requires_grad = True

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5,
    weight_decay=1e-4
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.3,
    patience=2
)

for epoch in range(FINE_TUNE_EPOCHS):
    start_time = time.time()

    print(
        f"\nFine-tuning Epoch {epoch + 1}/{FINE_TUNE_EPOCHS}",
        flush=True
    )

    # ---------------------------
    # Training with tqdm
    # ---------------------------
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    train_bar = tqdm(
        train_loader,
        desc=f"Train Epoch {epoch + 1}",
        unit="batch",
        dynamic_ncols=True
    )

    for images, labels in train_bar:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct += (outputs.argmax(1) == labels).sum().item()
        total += batch_size

        train_bar.set_postfix(
            loss=f"{running_loss / total:.4f}",
            acc=f"{correct / total:.2%}"
        )

    train_loss = running_loss / total
    train_acc = correct / total

    # ---------------------------
    # Validation with tqdm
    # ---------------------------
    model.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        valid_bar = tqdm(
            valid_loader,
            desc=f"Valid Epoch {epoch + 1}",
            unit="batch",
            dynamic_ncols=True
        )

        for images, labels in valid_bar:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            batch_size = labels.size(0)
            val_loss_sum += loss.item() * batch_size
            val_correct += (
                outputs.argmax(1) == labels
            ).sum().item()
            val_total += batch_size

            valid_bar.set_postfix(
                loss=f"{val_loss_sum / val_total:.4f}",
                acc=f"{val_correct / val_total:.2%}"
            )

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    # ---------------------------
    # Save best model
    # ---------------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())

        torch.save({
            "model_state_dict": best_state,
            "class_names": class_names,
            "num_classes": num_classes,
            "image_size": IMAGE_SIZE
        }, MODEL_PATH)

        status = "Best model saved!"
    else:
        status = "No improvement"

    elapsed = (time.time() - start_time) / 60

    print(f"\nEpoch {epoch + 1} Summary")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2%}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2%}")
    print(f"Time: {elapsed:.2f} minutes")
    print(status, flush=True)

# Restore best weights
model.load_state_dict(best_state)
model.eval()

print("\nFine-tuning completed!", flush=True)



Phase 2: Fine-tuning

Fine-tuning Epoch 1/5


Valid Epoch 1: 100%|██████████| 391/391 [00:28<00:00, 13.96batch/s, acc=97.01%, loss=0.0906]



Epoch 1 Summary
Train Loss: 0.1103
Train Accuracy: 96.32%
Validation Loss: 0.0906
Validation Accuracy: 97.01%
Time: 7.15 minutes
Best model saved!

Fine-tuning Epoch 2/5


Valid Epoch 2: 100%|██████████| 391/391 [00:40<00:00,  9.64batch/s, acc=97.73%, loss=0.0721]



Epoch 2 Summary
Train Loss: 0.0842
Train Accuracy: 97.26%
Validation Loss: 0.0721
Validation Accuracy: 97.73%
Time: 6.31 minutes
Best model saved!

Fine-tuning Epoch 3/5


...


Epoch 3 Summary
Train Loss: 0.0686
Train Accuracy: 97.74%
Validation Loss: 0.0625
Validation Accuracy: 98.08%
Time: 7.41 minutes
Best model saved!

Fine-tuning Epoch 4/5


Valid Epoch 4: 100%|██████████| 391/391 [00:40<00:00,  9.67batch/s, acc=98.38%, loss=0.0487]



Epoch 4 Summary
Train Loss: 0.0561
Train Accuracy: 98.15%
Validation Loss: 0.0487
Validation Accuracy: 98.38%
Time: 6.72 minutes
Best model saved!

Fine-tuning Epoch 5/5


Valid Epoch 5: 100%|██████████| 391/391 [00:32<00:00, 11.92batch/s, acc=98.68%, loss=0.0425]



Epoch 5 Summary
Train Loss: 0.0494
Train Accuracy: 98.28%
Validation Loss: 0.0425
Validation Accuracy: 98.68%
Time: 6.41 minutes
Best model saved!

Fine-tuning completed!


In [20]:

# --------------------------------------------------
# 9. Plot training history
# --------------------------------------------------

epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_acc"], label="Train")
plt.plot(epochs, history["val_acc"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_loss"], label="Train")
plt.plot(epochs, history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.savefig(PLOT_PATH, dpi=150)
plt.close()

print("\nTraining completed!")
print("Best model:", MODEL_PATH)
print("Class labels:", LABELS_PATH)
print("Training graph:", PLOT_PATH)



Training completed!
Best model: D:\PlantDiseaseDataset\saved_model\plant_disease_mobilenetv2.pth
Class labels: D:\PlantDiseaseDataset\saved_model\class_labels.json
Training graph: D:\PlantDiseaseDataset\saved_model\training_history.png
